# Email Spam Detection using Logistic Regression

## Prepared By: Sajid Ali

### Workflow
1. Load & explore the dataset
2. Preprocess text
3. Split into train/test sets
4. Build a TF-IDF + Logistic Regression pipeline
5. Cross-validate & train
6. Evaluate with metrics & visualizations
7. Inspect top spam/ham keywords
8. Predict on new emails

## Step 1 — Install & Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)
from sklearn.pipeline import Pipeline

## Step 2 — Load Dataset

In [ ]:
import pandas as pd

df = pd.read_csv("emails.csv")

df.head()

## Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('=== Dataset Info ===')
print(df.info())
print()
print('=== Missing Values ===')
print(df.isnull().sum())
print()
print('=== Label Distribution ===')
print(df[LABEL_COL].value_counts())
print(f'   Spam rate: {(df[LABEL_COL] == SPAM_VALUE).mean()*100:.1f}%')

In [ ]:
# Visualize label distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df[LABEL_COL].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Email Count by Label', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Spam vs Ham Ratio', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Email length analysis
df['text_length'] = df[TEXT_COL].astype(str).apply(len)
df['word_count']  = df[TEXT_COL].astype(str).apply(lambda x: len(x.split()))

print('=== Text Length Stats by Label ===')
print(df.groupby(LABEL_COL)[['text_length', 'word_count']].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for label, color in zip(df[LABEL_COL].unique(), ['#2ecc71', '#e74c3c']):
    subset = df[df[LABEL_COL] == label]
    axes[0].hist(subset['text_length'], bins=30, alpha=0.6, label=label, color=color, edgecolor='white')
    axes[1].hist(subset['word_count'],  bins=30, alpha=0.6, label=label, color=color, edgecolor='white')

axes[0].set_title('Character Length Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].set_title('Word Count Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Words')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 4 — Text Preprocessing

In [ ]:
def clean_text(text: str) -> str:
    """Lowercase, remove URLs, numbers, punctuation, collapse spaces."""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' url ', text)   # replace URLs
    text = re.sub(r'\d+', ' num ', text)                  # replace numbers
    text = re.sub(r'[^a-z\s]', ' ', text)                # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()              # collapse whitespace
    return text

# Drop nulls and apply cleaning
df = df[[TEXT_COL, LABEL_COL]].dropna().copy()
df['clean_text'] = df[TEXT_COL].apply(clean_text)

# Binary label: 1 = spam, 0 = ham
df['spam'] = (df[LABEL_COL].astype(str).str.lower() ==
              str(SPAM_VALUE).lower()).astype(int)

print('✅ Preprocessing complete. Sample:')
df[['text', 'clean_text', 'spam']].head(6)

## Step 5 — Train / Test Split

In [ ]:
X = df['clean_text']
y = df['spam']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y          # preserve spam ratio in both splits
)

print(f'📂 Training set : {len(X_train):,} emails  ({y_train.mean()*100:.1f}% spam)')
print(f'   Test set     : {len(X_test):,} emails  ({y_test.mean()*100:.1f}% spam)')

## Step 6 — Build Pipeline (TF-IDF → Logistic Regression)

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),     # unigrams + bigrams
        max_features=50_000,    # top 50k terms by TF-IDF score
        sublinear_tf=True,      # apply log(tf) scaling
        min_df=2,               # ignore terms appearing only once
        stop_words='english'
    )),
    ('clf', LogisticRegression(
        C=1.0,                  # inverse regularization strength
        solver='lbfgs',         # efficient for sparse data
        max_iter=1000,
        class_weight='balanced',# handles class imbalance
        random_state=RANDOM_STATE
    ))
])

print('✅ Pipeline built:')
print('   Step 1 → TfidfVectorizer (unigrams + bigrams, top 50k features)')
print('   Step 2 → LogisticRegression (lbfgs, balanced class weights)')

## Step 7 — Cross-Validation

In [ ]:
print('Running 5-fold cross-validation on training set …')

cv_f1  = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='f1',       n_jobs=-1)
cv_acc = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)

print(f'\n  F1 per fold      : {[f"{s:.3f}" for s in cv_f1]}')
print(f'  Mean F1          : {cv_f1.mean():.4f}  (± {cv_f1.std():.4f})')
print(f'  Mean Accuracy    : {cv_acc.mean()*100:.2f}%  (± {cv_acc.std()*100:.2f}%)')

# Plot CV scores
fig, ax = plt.subplots(figsize=(7, 4))
folds = [f'Fold {i+1}' for i in range(5)]
x = np.arange(5)
bars = ax.bar(x, cv_f1, color='steelblue', alpha=0.8, edgecolor='white', linewidth=1.5)
ax.axhline(cv_f1.mean(), color='crimson', linestyle='--', linewidth=2,
           label=f'Mean F1 = {cv_f1.mean():.3f}')
ax.set_xticks(x)
ax.set_xticklabels(folds)
ax.set_ylim(0, 1.05)
ax.set_ylabel('F1 Score')
ax.set_title('5-Fold Cross-Validation F1 Scores', fontsize=13, fontweight='bold')
ax.legend()
for bar, score in zip(bars, cv_f1):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{score:.3f}', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 8 — Train Final Model

In [ ]:
print('Training final model on full training set …')
pipeline.fit(X_train, y_train)
print('✅ Model trained successfully!')

## Step 9 — Evaluate on Test Set

In [ ]:
y_pred      = pipeline.predict(X_test)
y_pred_prob = pipeline.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc  = roc_auc_score(y_test, y_pred_prob)

print('=' * 50)
print('  TEST SET RESULTS')
print('=' * 50)
print(f'  Accuracy  : {accuracy*100:.2f}%')
print(f'  ROC-AUC   : {roc_auc:.4f}')
print()
print('  Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

## Step 10 — Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Confusion Matrix ──────────────────────────────
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'],
            annot_kws={'size': 16, 'weight': 'bold'})
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=11)
axes[0].set_ylabel('True Label', fontsize=11)

# Annotate with TP/TN/FP/FN labels
labels = [['TN', 'FP'], ['FN', 'TP']]
for i in range(2):
    for j in range(2):
        axes[0].text(j + 0.5, i + 0.75, labels[i][j],
                     ha='center', fontsize=9, color='gray')

# ── ROC Curve ─────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
axes[1].plot(fpr, tpr, color='steelblue', lw=2.5,
             label=f'Logistic Regression (AUC = {roc_auc:.3f})')
axes[1].fill_between(fpr, tpr, alpha=0.08, color='steelblue')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.02])
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

## Step 11 — Top Spam & Ham Keywords

In [ ]:
vectorizer    = pipeline.named_steps['tfidf']
clf           = pipeline.named_steps['clf']
feature_names = np.array(vectorizer.get_feature_names_out())
coef          = clf.coef_[0]

TOP_N = 20
top_spam_idx = coef.argsort()[-TOP_N:][::-1]
top_ham_idx  = coef.argsort()[:TOP_N]

spam_words   = feature_names[top_spam_idx]
spam_weights = coef[top_spam_idx]
ham_words    = feature_names[top_ham_idx]
ham_weights  = coef[top_ham_idx]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Spam keywords
axes[0].barh(spam_words[::-1], spam_weights[::-1], color='#e74c3c', alpha=0.85, edgecolor='white')
axes[0].set_title(f'Top {TOP_N} SPAM Keywords', fontsize=13, fontweight='bold', color='#e74c3c')
axes[0].set_xlabel('Logistic Regression Weight')
axes[0].axvline(0, color='black', linewidth=0.8)

# Ham keywords
axes[1].barh(ham_words, np.abs(ham_weights), color='#2ecc71', alpha=0.85, edgecolor='white')
axes[1].set_title(f'Top {TOP_N} HAM Keywords', fontsize=13, fontweight='bold', color='#27ae60')
axes[1].set_xlabel('|Logistic Regression Weight|')

plt.tight_layout()
plt.show()

print(f'\n🔑 Top {TOP_N} SPAM keywords:')
for w, s in zip(spam_words, spam_weights):
    print(f'   {w:<30} weight = {s:+.4f}')

print(f'\n🔑 Top {TOP_N} HAM keywords:')
for w, s in zip(ham_words, ham_weights):
    print(f'   {w:<30} weight = {s:+.4f}')

## Step 12 — Predict on New Emails

In [ ]:
sample_emails = [
    "Congratulations! You've won a FREE iPhone. Click here to claim your prize now!",
    "Hey, are we still meeting for lunch tomorrow at 12?",
    "URGENT: Your bank account has been suspended. Verify now to avoid charges.",
    "Please find the attached quarterly report for your review.",
    "Win $1000 cash prize! Send your details to claim immediately!",
    "Can you please review the pull request I submitted this morning?",
    "Your PayPal account is limited! Log in now or it will be closed permanently!",
    "The team lunch is rescheduled to Thursday at 1 PM. Hope that works for everyone."
]

cleaned       = [clean_text(e) for e in sample_emails]
preds         = pipeline.predict(cleaned)
probs         = pipeline.predict_proba(cleaned)[:, 1]

results = pd.DataFrame({
    'Email'       : [e[:75] + ('…' if len(e) > 75 else '') for e in sample_emails],
    'Prediction'  : ['🚨 SPAM' if p == 1 else '✅ HAM' for p in preds],
    'Spam Prob %' : [f'{p*100:.1f}%' for p in probs]
})

print('=== Predictions on Sample Emails ===')
display(results)

## Step 13 — Try Your Own Email!

In [ ]:
# ✏️ Type any email here and run this cell to classify it
my_email = "You have been selected to receive a $500 gift card. Click here to claim!"

cleaned_email = clean_text(my_email)
pred          = pipeline.predict([cleaned_email])[0]
prob          = pipeline.predict_proba([cleaned_email])[0][1]

label = '🚨 SPAM' if pred == 1 else '✅ HAM'
print(f'Email   : "{my_email}"')
print(f'Result  : {label}')
print(f'Spam probability: {prob*100:.1f}%')

---
## 📝 Summary

| Step | Details |
|------|---------|
| **Dataset** | `emails.csv` — 200 emails (100 spam, 100 ham) |
| **Preprocessing** | Lowercasing, URL/number/punctuation removal |
| **Features** | TF-IDF with unigrams + bigrams (top 50k features) |
| **Model** | Logistic Regression (lbfgs, balanced class weights) |
| **Validation** | 5-fold cross-validation (F1 score) |
| **Metrics** | Accuracy, ROC-AUC, Precision, Recall, F1 |

> **Why Logistic Regression?**  
> It is the standard *linear* model for binary classification. It outputs probabilities via the sigmoid function, handles high-dimensional sparse TF-IDF features well, and its weights are directly interpretable as word importance scores.
